In [1]:
import warnings

import numpy as np
import pandas as pd
from scipy import stats

from adaptive_behaviour import paths

warnings.filterwarnings("ignore")
pd.set_option("display.width", 130)

# Analytic maxima. Entropy is summed over the constituent distributions of each parameter, so the
# maximum is (number of distributions) x log(support size), attained at the uniform distribution.
N_BALL_X, N_BALL_Y, N_PADDLE, N_ACTIONS = 38, 8, 8, 3
N_STATES = N_BALL_X * N_BALL_Y * N_PADDLE

# CL is a (numA x numS) mapping: one distribution over actions per state.
CL_MAX = N_STATES * np.log(N_ACTIONS)

# B[f] for factor f has shape (n_f, n_f, n_actions_f): one distribution over next-states
# per (state, action) pair.
B_MAX = {
    1: N_BALL_X * N_ACTIONS * np.log(N_BALL_X),   # ball-x, 3 controllable actions
    2: N_BALL_Y * 1 * np.log(N_BALL_Y),           # ball-y, uncontrolled
    3: N_PADDLE * 1 * np.log(N_PADDLE),           # paddle-y, uncontrolled
}
# C[m] is one distribution over observations per modality.
C_MAX = {1: np.log(N_BALL_X), 2: np.log(N_BALL_Y), 3: np.log(N_PADDLE)}

print(f"CL maximum (uniform) = {N_STATES} x ln {N_ACTIONS} = {CL_MAX:.2f}")
for k, v in B_MAX.items():
    print(f"B factor {k} maximum   = {v:.2f}")

CL maximum (uniform) = 2432 x ln 3 = 2671.83
B factor 1 maximum   = 414.68
B factor 2 maximum   = 16.64
B factor 3 maximum   = 16.64


In [2]:
CL_FILES = {f"CFL-{t}": paths.RESULTS_CL / f"c_data_M{t}.csv" for t in (1, 2, 3, 4)}
B_FILES  = {f"DP-{t}":  paths.RESULTS_DPEFE / f"b_data_{t}.csv" for t in (2, 5, 10, 20)}
B_FILES["AIF-1"] = paths.RESULTS_SI / "b_data.csv"

rows = []
for label, path in CL_FILES.items():
    d = pd.read_csv(path)
    lo, hi = d.entropy_1.min(), d.entropy_1.max()
    rows.append(dict(agent=label, parameter="CL", maximum=CL_MAX,
                     observed_max=hi, observed_min=lo,
                     frac_of_max_start=hi / CL_MAX, frac_of_max_end=lo / CL_MAX,
                     total_drop_pct=(hi - lo) / CL_MAX * 100))

for label, path in B_FILES.items():
    try:
        d = pd.read_csv(path)
    except FileNotFoundError:
        print(f"  (skipping {label}: {path} not found)")
        continue
    lo, hi = d.entropy_1.min(), d.entropy_1.max()
    rows.append(dict(agent=label, parameter="B (ball-x)", maximum=B_MAX[1],
                     observed_max=hi, observed_min=lo,
                     frac_of_max_start=hi / B_MAX[1], frac_of_max_end=lo / B_MAX[1],
                     total_drop_pct=(hi - lo) / B_MAX[1] * 100))

raw = pd.DataFrame(rows)
raw.round(3)

,agent,parameter,maximum,observed_max,observed_min,frac_of_max_start,frac_of_max_end,total_drop_pct
0,CFL-1,CL,2671.825,2671.820,2640.255,1.000,0.988,1.181
1,CFL-2,CL,2671.825,2671.818,2618.016,1.000,0.980,2.014
2,CFL-3,CL,2671.825,2671.816,2594.121,1.000,0.971,2.908
3,CFL-4,CL,2671.825,2671.696,2557.002,1.000,0.957,4.293
4,DP-2,B (ball-x),414.685,324.077,28.425,0.782,0.069,71.296
5,DP-5,B (ball-x),414.685,324.077,40.926,0.782,0.099,68.281
6,DP-10,B (ball-x),414.685,324.077,42.029,0.782,0.101,68.015
7,DP-20,B (ball-x),414.685,324.077,42.029,0.782,0.101,68.015
8,AIF-1,B (ball-x),414.685,354.518,6.917,0.855,0.017,83.823


In [3]:
def nte_curve(path, maximum, col="entropy_1"):
    d = pd.read_csv(path)
    d["half"] = d.half.astype(int)
    d["nte"] = d[col] / maximum
    return d


print("Absolute NTE, mean over seeds, by time window")
print("(1.0 = uniform / uninformed; lower = more informed)\n")
summary = []
for label, path in CL_FILES.items():
    d = nte_curve(path, CL_MAX)
    g = d.groupby("half").nte.mean()
    summary.append(dict(agent=label, parameter="CL", w1=g.get(0), w2=g.get(1),
                        change=g.get(1) - g.get(0)))
for label, path in B_FILES.items():
    try:
        d = nte_curve(path, B_MAX[1])
    except FileNotFoundError:
        continue
    g = d.groupby("half").nte.mean()
    summary.append(dict(agent=label, parameter="B (ball-x)", w1=g.get(0), w2=g.get(1),
                        change=g.get(1) - g.get(0)))

pd.DataFrame(summary).round(4)

Absolute NTE, mean over seeds, by time window
(1.0 = uniform / uninformed; lower = more informed)



,agent,parameter,w1,w2,change
0,CFL-1,CL,0.9987,0.9956,-0.0031
1,CFL-2,CL,0.9976,0.9922,-0.0054
2,CFL-3,CL,0.9960,0.9884,-0.0076
3,CFL-4,CL,0.9936,0.9814,-0.0122
4,DP-2,B (ball-x),0.2758,0.2177,-0.0580
5,DP-5,B (ball-x),0.3566,0.3159,-0.0407
6,DP-10,B (ball-x),0.3584,0.3239,-0.0345
7,DP-20,B (ball-x),0.3570,0.3187,-0.0383
8,AIF-1,B (ball-x),0.4148,0.3616,-0.0532


In [4]:
BEHAVIOUR = {f"CFL-{t}": paths.RESULTS_CL / f"data_clmethod_M{t}.csv" for t in (1, 2, 3, 4)}
BEHAVIOUR.update({f"DP-{t}": paths.RESULTS_DPEFE / f"p_data_{t}.csv" for t in (2, 5, 10, 20)})
BEHAVIOUR["AIF-1"] = paths.RESULTS_SI / "p_data.csv"

ENTROPY = dict(CL_FILES)
ENTROPY.update(B_FILES)

rows = []
for agent, bpath in BEHAVIOUR.items():
    epath = ENTROPY.get(agent)
    if epath is None:
        continue
    try:
        b = pd.read_csv(bpath); e = pd.read_csv(epath)
    except FileNotFoundError:
        continue
    b["half"] = b.half.astype(int); e["half"] = e.half.astype(int)

    bs = b.groupby(["session_num", "half"]).hit_count.mean().unstack()
    es = e.groupby(["session_num", "half"]).entropy_1.mean().unstack()
    m = pd.DataFrame({"d_behaviour": bs[1] - bs[0], "d_entropy": es[1] - es[0]}).dropna()
    if len(m) < 5:
        continue
    r, p = stats.pearsonr(m.d_entropy, m.d_behaviour)
    rho, prho = stats.spearmanr(m.d_entropy, m.d_behaviour)
    rows.append(dict(agent=agent, n_seeds=len(m), pearson_r=r, p=p,
                     spearman_rho=rho, p_spearman=prho))

pred = pd.DataFrame(rows)
pred["predicts"] = np.where(pred.p < 0.05, "yes", "no")
print("Across-seed correlation between entropy change and behavioural change")
print("(negative r = seeds that became more informed improved more)\n")
pred.round(4)

Across-seed correlation between entropy change and behavioural change
(negative r = seeds that became more informed improved more)



,agent,n_seeds,pearson_r,p,spearman_rho,p_spearman,predicts
0,CFL-1,100,-0.2052,0.0406,-0.1216,0.2282,yes
1,CFL-2,100,-0.1680,0.0947,-0.0075,0.9406,no
2,CFL-3,100,-0.0536,0.5962,0.0239,0.8135,no
3,CFL-4,100,-0.1192,0.2375,0.0396,0.6960,no
4,DP-2,99,0.0699,0.4921,0.0152,0.8817,no
5,DP-5,99,0.0419,0.6808,0.0209,0.8377,no
6,DP-10,99,0.1951,0.0529,0.2030,0.0439,no
7,DP-20,99,-0.0971,0.3391,-0.0945,0.3523,no
8,AIF-1,99,0.0616,0.5449,0.0277,0.7852,no


In [ ]:
from adaptive_behaviour.pong.analysis.tables import nte_tables as N

# The `predicts` column above is uncorrected. This is a family of nine tests of one question, and
# the manuscript applies Benjamini-Hochberg everywhere else, so the supplementary table reports
# corrected values. The correction matters here: CFL-1 is nominally significant at p = 0.041 and
# is not once corrected, and its Pearson and Spearman coefficients disagree in any case.
res = N.write_tables()

print("Entropy against the analytic maximum (Supplementary Table S7)")
print(res["entropy"][["agent", "parameter", "w1", "w2", "change", "drop_pct"]]
      .round(4).to_string(index=False))

print("\nDoes entropy change predict behaviour? (Supplementary Table S8)")
print(res["prediction"][["agent", "n", "r", "p_adj", "rho", "p_rho_adj"]]
      .round(4).to_string(index=False))
print(f"\nsmallest corrected p over all 18 tests: {min(res['prediction'].p_adj.min(), res['prediction'].p_rho_adj.min()):.3f}")
print("wrote tables/tab_nte.tex and tables/tab_nte_predict.tex")